# colab6 — Fixed Architecture: w_i IS y_i

## The Bug
Paper says: constant 1 × w_i → y_i (w_i IS the character)
Code does: y_char × w_i → y_char*w_i (w_i is a scaling factor on a known value)

## The Fix
Feed constant 1 into for_gen_dense instead of y from training data.

| Exp | Encoding | Target | Expected w | Converge? |
|---|---|---|---|---|
| 1 | {1,2} | 11 | [1,1] | Yes |
| 2 | {1,2} | 21 | [2,1] | Yes |
| 3 | {1,2} | 21211 | [2,1,2,1,1] | Yes |
| 4 | {0,1} | 01 | [0,1] | No (ReLU dead zone) |
| 5 | {0,1} | 11 | [1,1] | Yes |


In [ ]:
!pip install tensorflow ml_dtypes --upgrade -q

In [ ]:
import random, time, math, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Input, Lambda, concatenate
print(f"TF {tf.__version__}")


## Architecture (matching, min, minimum modules — unchanged)

In [ ]:
def transform_seqs_to_input(seqA, seqB):
    matching_pairs = []
    matching_pairs.append([int(seqA[0]), int(seqB[0])])
    if len(seqA) == 1 and len(seqB) == 1:
        return matching_pairs
    input_length_x = len(seqA)
    match_layers_i = (input_length_x * 2) - 1
    start_i, end_i = 1, 2
    for l in range(match_layers_i):
        if l < input_length_x - 1:
            i, j = [*reversed(range(0, end_i))], [*range(0, end_i)]
            for n in range(len(i)):
                if j[n] < len(seqB):
                    matching_pairs.append([int(seqA[i[n]]), int(seqB[j[n]])])
            end_i += 1
        else:
            i, j = [*reversed(range(start_i, input_length_x))], [*range(start_i, input_length_x)]
            for n in range(len(i)):
                if j[n] < len(seqB):
                    matching_pairs.append([int(seqA[i[n]]), int(seqB[j[n]])])
            start_i += 1
            if start_i > len(seqB): break
    return matching_pairs

def transform_input_for_generate(inp):
    x, y = [], []
    for pair in inp:
        x.append(pair[0]); y.append(pair[1])
    return [x, y]

def matching_module():
    epsilon = 1
    model = Sequential()
    model.add(Dense(2, activation='relu', use_bias=True, input_shape=(2,)))
    model.add(Dense(2, activation='relu', use_bias=True))
    model.add(Dense(1, activation='relu', use_bias=True))
    w1=model.layers[0].get_weights(); w1[0][0][0],w1[0][0][1]=1.,-1.; w1[0][1][0],w1[0][1][1]=-1.,1.; w1[1][0],w1[1][1]=0,0
    w2=model.layers[1].get_weights(); w2[0][0][0],w2[0][0][1]=1.,1.; w2[0][1][0],w2[0][1][1]=1.,1.; w2[1][0],w2[1][1]=epsilon,-epsilon
    w3=model.layers[2].get_weights(); w3[0][0][0],w3[0][1][0]=1./epsilon,-1./epsilon; w3[1][0]=-1
    model.layers[0].set_weights(w1); model.layers[1].set_weights(w2); model.layers[2].set_weights(w3)
    model.trainable = False
    return model

def min_module(i, j, k):
    inp = Input(shape=(2,))
    x = Dense(2, activation='relu', use_bias=True)(inp)
    combined = concatenate([x, inp])
    z = Dense(1, activation='relu', use_bias=True, name=f'result_pixel_{i}{j}_{k}')(combined)
    model = Model(inputs=inp, outputs=z)
    w1=model.layers[1].get_weights(); w1[0][0],w1[0][1]=[-1.,1.],[1.,-1.]
    w2=model.layers[3].get_weights(); w2[0][0],w2[0][1],w2[0][2],w2[0][3]=-0.5,-0.5,0.5,0.5
    model.layers[1].set_weights(w1); model.layers[3].set_weights(w2)
    model.trainable = False
    return model

def minimum(i, j):
    inp = Input(shape=(3,))
    c1 = Lambda(lambda x: x[:,:2])(inp)
    c2 = Lambda(lambda x: x[:,2:])(inp)
    m = min_module(i,j,1)(c1)
    output = min_module(i,j,2)(concatenate([c2, m]))
    model = Model(inputs=inp, outputs=output)
    model.trainable = False
    return model


## THE FIX: `align_model_for_N_FIXED`

`for_gen_dense` receives constant 1 instead of y from data. **w_i IS y_i.**

In [ ]:
def align_model_for_N_FIXED(seq_length_x, seq_length_y, matching_pair_number):
    inp = Input(shape=(2, matching_pair_number), name='input')
    x = Lambda(lambda t: t[:, 0, :])(inp)

    # THE FIX: constant 1 input for learnable y characters
    const_one = Lambda(lambda t: tf.ones((tf.shape(t)[0], 1)))(inp)

    out = {}
    for i in range(seq_length_y):
        layername = f'for_gen_dense_{i+1}'
        out[layername] = Dense(1, activation='relu', name=layername, use_bias=False)(const_one)

    # Rest is identical to original
    pair_i = 1; calc_layer = (seq_length_x * 2) - 1; test_dict = {}
    densed_y = out['for_gen_dense_1']
    x_char = Lambda(lambda t: t[:, 0:1])(x)
    pair_11 = concatenate([x_char, densed_y], name='matching_debug_1')
    ext_gaps = Dense(2, activation='relu', name='first_calc_gap_layer')(pair_11)
    min1 = min_module(1,1,1)(ext_gaps)
    matching1 = matching_module()(pair_11)
    z = min_module(1,1,2)(concatenate([min1, matching1]))
    result_pixel_11 = concatenate([ext_gaps, z], name='input_pixel_1_1')
    pair_i = 2
    if seq_length_x == 1 and seq_length_y == 1:
        return Model(inputs=inp, outputs=z)

    test_dict['input_pixel_1_1'] = result_pixel_11
    test_dict['result_pixel_1_1'] = z
    comp_i_val, comp_j_val = 1, 2
    start_sentinel, end_sentinel = 1, 2
    unbalance_flag = True

    for calc_layer_i in range(calc_layer):
        if calc_layer_i < seq_length_x - 1:
            comp_i_val, comp_j_val = start_sentinel, end_sentinel
            while comp_i_val <= end_sentinel:
                if comp_i_val <= seq_length_y:
                    il = f'input_{comp_i_val}_{comp_j_val}'
                    bil = f'before_input_{comp_i_val}_{comp_j_val}'
                    c = pair_i
                    densed_y = out[f'for_gen_dense_{comp_i_val}']
                    x_char = Lambda(lambda t, c=c: t[:, c-1:c])(x)
                    pair = concatenate([x_char, densed_y], name=f'matching_debug_{c}')
                    matching = matching_module()(pair)
                    if comp_i_val == 1:
                        pi = test_dict[f'input_pixel_{comp_i_val}_{comp_j_val-1}']
                        pr = test_dict[f'result_pixel_{comp_i_val}_{comp_j_val-1}']
                        g = Lambda(lambda t: t[:, 0:1])(pi)
                        bi = concatenate([g, pr, matching], name=bil)
                    elif comp_j_val == 1:
                        pi = test_dict[f'input_pixel_{comp_i_val-1}_{comp_j_val}']
                        pr = test_dict[f'result_pixel_{comp_i_val-1}_{comp_j_val}']
                        g = Lambda(lambda t: t[:, 0:1])(pi)
                        bi = concatenate([g, pr, matching], name=bil)
                    else:
                        pr1 = test_dict[f'result_pixel_{comp_i_val}_{comp_j_val-1}']
                        pr2 = test_dict[f'result_pixel_{comp_i_val-1}_{comp_j_val}']
                        pr3 = test_dict[f'result_pixel_{comp_i_val-1}_{comp_j_val-1}']
                        bi = concatenate([pr1, pr2, pr3, matching], name=bil)
                    ip = Dense(3, activation='relu', name=il)(bi)
                    rp = minimum(comp_i_val, comp_j_val)(ip)
                    test_dict[f'input_pixel_{comp_i_val}_{comp_j_val}'] = ip
                    test_dict[f'result_pixel_{comp_i_val}_{comp_j_val}'] = rp
                    if unbalance_flag: unbalance_flag = False
                comp_i_val += 1; comp_j_val -= 1; pair_i += 1
                if unbalance_flag: pair_i -= 1
                unbalance_flag = True
            if end_sentinel + 1 <= seq_length_x: end_sentinel += 1
        else:
            start_sentinel += 1
            comp_i_val, comp_j_val = start_sentinel, end_sentinel
            while comp_i_val <= end_sentinel:
                if comp_i_val <= seq_length_y:
                    bil = f'before_input_{comp_i_val}_{comp_j_val}'
                    il = f'input_{comp_i_val}_{comp_j_val}'
                    c = pair_i
                    densed_y = out[f'for_gen_dense_{comp_i_val}']
                    x_char = Lambda(lambda t, c=c: t[:, c-1:c])(x)
                    pair = concatenate([x_char, densed_y], name=f'matching_debug_{c}')
                    matching = matching_module()(pair)
                    pr1 = test_dict[f'result_pixel_{comp_i_val}_{comp_j_val-1}']
                    pr2 = test_dict[f'result_pixel_{comp_i_val-1}_{comp_j_val}']
                    pr3 = test_dict[f'result_pixel_{comp_i_val-1}_{comp_j_val-1}']
                    bi = concatenate([pr1, pr2, pr3, matching], name=bil)
                    ip = Dense(3, activation='relu', name=il)(bi)
                    rp = minimum(comp_i_val, comp_j_val)(ip)
                    test_dict[f'input_pixel_{comp_i_val}_{comp_j_val}'] = ip
                    test_dict[f'result_pixel_{comp_i_val}_{comp_j_val}'] = rp
                    if unbalance_flag: unbalance_flag = False
                comp_i_val += 1; comp_j_val -= 1; pair_i += 1
                if unbalance_flag: pair_i -= 1
                unbalance_flag = True
                if start_sentinel == end_sentinel:
                    return Model(inputs=inp, outputs=rp)


## Weight init + freezing

In [ ]:
def set_weight_for_debug(model, sx, sy, mp):
    print('setting frozen weights ...')
    w=model.get_layer('first_calc_gap_layer').get_weights()
    w[0][0][0],w[0][0][1]=0,0; w[0][1][0],w[0][1][1]=0,0; w[1][0],w[1][1]=2,2
    model.get_layer('first_calc_gap_layer').set_weights(w)
    model.get_layer('first_calc_gap_layer').trainable=False
    if sx <= 1: return
    calc_layer=(sx*2)-1; comp_i,comp_j=1,2; start_s,end_s=1,2
    for cli in range(calc_layer):
        if cli < sx-1:
            comp_i,comp_j=start_s,end_s
            while comp_i<=end_s:
                if comp_i<=sy:
                    il=f'input_{comp_i}_{comp_j}'
                    w=model.get_layer(il).get_weights()
                    if comp_i==1 or comp_j==1:
                        w[0][0][0],w[0][0][1],w[0][0][2]=1,0,1
                        w[0][1][0],w[0][1][1],w[0][1][2]=0,1,0
                        w[0][2][0],w[0][2][1],w[0][2][2]=0,0,1
                        w[1][0],w[1][1],w[1][2]=1,1,-1
                    else:
                        w[0][0][0],w[0][0][1],w[0][0][2]=1,0,0
                        w[0][1][0],w[0][1][1],w[0][1][2]=0,1,0
                        w[0][2][0],w[0][2][1],w[0][2][2]=0,0,1
                        w[0][3][0],w[0][3][1],w[0][3][2]=0,0,1
                        w[1][0],w[1][1],w[1][2]=1,1,0
                    model.get_layer(il).set_weights(w)
                    model.get_layer(il).trainable=False
                comp_i,comp_j=comp_i+1,comp_j-1
            if end_s+1<=sx: end_s+=1
        else:
            start_s+=1; comp_i,comp_j=start_s,end_s
            while comp_i<=end_s:
                if comp_i<=sy:
                    il=f'input_{comp_i}_{comp_j}'
                    w=model.get_layer(il).get_weights()
                    w[0][0][0],w[0][0][1],w[0][0][2]=1,0,0
                    w[0][1][0],w[0][1][1],w[0][1][2]=0,1,0
                    w[0][2][0],w[0][2][1],w[0][2][2]=0,0,1
                    w[0][3][0],w[0][3][1],w[0][3][2]=0,0,1
                    w[1][0],w[1][1],w[1][2]=1,1,0
                    model.get_layer(il).set_weights(w)
                    model.get_layer(il).trainable=False
                comp_i,comp_j=comp_i+1,comp_j-1
                if start_s==end_s: break

def froozen_align_model(model):
    print('freezing non-trainable layers ...')
    for layer in model.layers:
        if 'for_gen_dense' not in layer.name:
            layer.trainable = False


## Helpers: Levenshtein + CSV generation + Verification

In [ ]:
def levenshtein_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0]=i
    for j in range(n+1): dp[0][j]=j
    for i in range(1,m+1):
        for j in range(1,n+1):
            cost = 0 if s1[i-1]==s2[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[m][n]

def generate_csv(target, alphabet, path):
    n = len(target)
    strs = []
    def gen(p):
        if len(p)==n: strs.append(p); return
        for c in alphabet: gen(p+c)
    gen('')
    with open(path,'w') as f:
        for s in strs: f.write(f'{s},{target},{levenshtein_distance(s,target)}\n')
    print(f'{len(strs)} examples -> {path}')

def verify_model(model, csv_path, target_str):
    print(f'Verify: target={target_str}')
    for i,ch in enumerate(target_str):
        w=model.get_layer(f'for_gen_dense_{i+1}').get_weights()
        w[0][0][0]=float(ch)
        model.get_layer(f'for_gen_dense_{i+1}').set_weights(w)
    with open(csv_path) as f: lines=[l.strip() for l in f]
    ok=True
    for line in lines:
        sp=line.split(','); xs,ys,td=sp[0],sp[1],int(sp[2])
        pairs=transform_seqs_to_input(xs,ys)
        inp=transform_input_for_generate(pairs)
        pred=model(tf.constant([inp],dtype=tf.float32),training=False).numpy()[0][0]
        if abs(pred-td)>0.01: ok=False; print(f'  WRONG: x={xs} true={td} pred={pred:.3f}')
    print(f'  {"ALL CORRECT" if ok else "ERRORS"}')
    return ok


## Training + plotting

In [ ]:
def training_fixed(target_str, csv_path, encoding='relabel',
                   epochs=50, lr=0.1, opt='sgd', seed=None):
    if seed:
        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    LEN=len(target_str); tw=[float(c) for c in target_str]
    with open(csv_path) as f: lines=[l.strip() for l in f]
    print(f'\nTarget={target_str} weights={tw} enc={encoding} {opt} lr={lr} epochs={epochs} samples={len(lines)}')

    sp=lines[0].split(','); x_s,y_s=sp[0],sp[1]
    pairs=transform_seqs_to_input(x_s,y_s)
    SX,SY,PL=len(x_s),len(y_s),len(pairs)
    model=align_model_for_N_FIXED(SX,SY,PL)
    set_weight_for_debug(model,SX,SY,PL)
    froozen_align_model(model)
    verify_model(model,csv_path,target_str)

    lo,hi=(0.5,2.5) if encoding=='relabel' else (0.0,1.5)
    iw=[]
    for i in range(LEN):
        w=model.get_layer(f'for_gen_dense_{i+1}').get_weights()
        w[0][0][0]=random.uniform(lo,hi)
        model.get_layer(f'for_gen_dense_{i+1}').set_weights(w)
        iw.append(float(w[0][0][0]))
    print(f'Init: {[f"{v:.3f}" for v in iw]}')

    pw=[list(iw)]; pl=[]
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr) if opt=='adam' else tf.keras.optimizers.SGD(learning_rate=lr)
    loss_fn=tf.keras.losses.MeanSquaredError()

    for epoch in range(epochs):
        loss=tf.Variable(0.0)
        with tf.GradientTape() as tape:
            for line in lines:
                sp=line.split(','); xs,ys,ts=sp[0],sp[1],int(sp[2])
                inp=transform_input_for_generate(transform_seqs_to_input(xs,ys))
                logit=model(tf.constant([inp],dtype=tf.float32),training=True)
                loss=loss+loss_fn(ts,logit)
            bl=loss/len(lines)
            grads=tape.gradient(bl,model.trainable_weights)
            optimizer.apply_gradients(zip(grads,model.trainable_weights))
        pl.append(float(bl.numpy()))
        cur=[float(model.get_layer(f'for_gen_dense_{i+1}').get_weights()[0][0][0]) for i in range(LEN)]
        pw.append(cur)
        if epoch%10==0 or epoch==epochs-1:
            print(f'  ep {epoch:3d} | loss={float(bl.numpy()):.6f} | w={[f"{v:.4f}" for v in cur]}')

    fw=[float(model.get_layer(f'for_gen_dense_{i+1}').get_weights()[0][0][0]) for i in range(LEN)]
    me=max(abs(f-t) for f,t in zip(fw,tw))
    print(f'Final: {[f"{v:.4f}" for v in fw]}  Target: {tw}  MaxErr: {me:.4f} {"OK" if me<0.1 else "FAIL"}')
    return {'target_str':target_str,'target_weights':tw,'init_weights':iw,
            'final_weights':fw,'progress_weights':pw,'progress_losses':pl,'encoding':encoding}

def plot_it(r, title=''):
    pw=np.array(r['progress_weights']); ls=r['progress_losses']; tw=r['target_weights']
    fig,ax=plt.subplots(1,2,figsize=(14,5))
    colors=plt.rcParams['axes.prop_cycle'].by_key()['color']
    for i in range(pw.shape[1]):
        ax[0].plot(pw[:,i],marker='o',ms=2,label=f'w{i+1}(tgt={tw[i]:.0f})',color=colors[i%len(colors)])
    for t in sorted(set(tw)): ax[0].axhline(t,color='grey',ls='--',alpha=.5)
    ax[0].set_xlabel('epoch');ax[0].set_ylabel('weight');ax[0].legend(fontsize=8)
    ax[0].set_title('Weights -> target char values')
    ax[1].plot(range(1,len(ls)+1),ls,'r-o',ms=2);ax[1].set_xlabel('epoch');ax[1].set_ylabel('MSE')
    ax[1].set_ylim(bottom=0);ax[1].set_title('Loss')
    fig.suptitle(title,fontsize=13,fontweight='bold');plt.tight_layout();plt.show()


## Generate data

In [ ]:
generate_csv('11',['1','2'],'data_r_11.csv')
generate_csv('21',['1','2'],'data_r_21.csv')
generate_csv('21211',['1','2'],'data_r_21211.csv')
generate_csv('01',['0','1'],'data_o_01.csv')
generate_csv('11',['0','1'],'data_o_11.csv')


## Exp 1: {1,2} target="11" (baseline)

In [ ]:
r1=training_fixed('11','data_r_11.csv',encoding='relabel',epochs=50,seed=42)
plot_it(r1,'Exp1: {1,2} target=11')

## Exp 2: {1,2} target="21" — THE KEY TEST
Weights should → **[2, 1]**, not [1, 1]!

In [ ]:
r2=training_fixed('21','data_r_21.csv',encoding='relabel',epochs=50,seed=42)
plot_it(r2,'Exp2: {1,2} target=21 (relabeled 01)')

## Exp 3: {1,2} target="21211" (hard 5-char case)
Weights should → **[2, 1, 2, 1, 1]**

In [ ]:
r3=training_fixed('21211','data_r_21211.csv',encoding='relabel',epochs=50,seed=42)
plot_it(r3,'Exp3: {1,2} target=21211')

## Exp 4: {0,1} target="01" — EXPECT FAILURE
ReLU kills gradient when w approaches 0.

In [ ]:
r4=training_fixed('01','data_o_01.csv',encoding='original',epochs=50,seed=42)
plot_it(r4,'Exp4: {0,1} target=01 (expect failure)')

## Exp 5: {0,1} target="11" (no zeros — should work)

In [ ]:
r5=training_fixed('11','data_o_11.csv',encoding='original',epochs=50,seed=42)
plot_it(r5,'Exp5: {0,1} target=11')

## Exp 6: {1,2} target="21211" + Adam

In [ ]:
r6=training_fixed('21211','data_r_21211.csv',encoding='relabel',epochs=100,lr=0.05,opt='adam',seed=42)
plot_it(r6,'Exp6: {1,2} 21211 + Adam')

## Summary

In [ ]:
print('\n'+'='*70+'\nSUMMARY\n'+'='*70)
for name,r in [('Exp1:{1,2} 11',r1),('Exp2:{1,2} 21',r2),('Exp3:{1,2} 21211',r3),
               ('Exp4:{0,1} 01',r4),('Exp5:{0,1} 11',r5),('Exp6:{1,2} 21211+Adam',r6)]:
    me=max(abs(f-t) for f,t in zip(r['final_weights'],r['target_weights']))
    print(f'  {name:25s} | loss={r["progress_losses"][-1]:.6f} | err={me:.4f} | {"OK" if me<0.1 else "FAIL"}')
    print(f'    target={r["target_weights"]}  learned={[round(w,3) for w in r["final_weights"]]}')


## What to look for

- **Exp 2**: weights → [2, 1] (not [1, 1] like the buggy version). This proves the fix works.
- **Exp 3**: weights → [2, 1, 2, 1, 1]. The hard case that the paper couldn't solve.
- **Exp 4 vs Exp 2**: Same underlying string "01", different encodings. {0,1} should fail, {1,2} should succeed.
- **Exp 4 vs Exp 5**: Both {0,1}, but target "01" fails while "11" works — confirming the zero-gradient problem.

If these predictions hold, the paper's diagnosis of "local solutions" was wrong. The real issues were:
1. **Code bug**: target string leaked into input (w_i was a scaling factor, not the character)
2. **ReLU dead zone**: character value 0 kills gradients through ReLU

Both are fixed by (1) feeding constant 1 into for_gen_dense and (2) relabeling {0,1} → {1,2}.
